# 🌲 Training XGBoost for Satellite Soil Nutrient Prediction
### Spatial Block Cross-Validation (Eliminating Spatial Autocorrelation Leakage)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)
[![XGBoost](https://img.shields.io/badge/Model-XGBoost%20Regressor-orange.svg)](https://xgboost.readthedocs.io/)

---

### 📌 Purpose of this Notebook:
This notebook trains and evaluates **XGBoost Regressors** to predict primary soil chemical nutrients:
- **Soil pH**
- **Available Nitrogen (N)**
- **Extractable Phosphorus (P)**
- **Exchangeable Potassium (K)**

### 🛡️ Why Spatial Block Cross-Validation?
Standard random `train_test_split` causes **spatial data leakage** because test points adjacent to training points share the same local soil and weather, leading to falsely inflated metrics.  
This notebook implements **Spatial GroupKFold**, holding out entire geographic blocks (e.g. administrative regions or $2^\circ 	imes 2^\circ$ blocks) exclusively for testing to evaluate true regional generalization.


## 📦 Step 1: Libraries & Environment Setup

In [ ]:
# Install and import libraries
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb

print(f"✅ XGBoost version: {xgb.__version__}")


## 📂 Step 2: Load Dataset (`Model_A.csv` or Generated Dataset)
Upload **`Model_A.csv`** or **`lucas_sentinel2_soil_dataset.csv`** using the Files panel on the left.


In [ ]:
# Search for dataset
possible_files = ["Model_A.csv", "/content/Model_A.csv", "lucas_sentinel2_soil_dataset.csv", "/content/lucas_sentinel2_soil_dataset.csv"]
dataset_path = None

for f in possible_files:
    if os.path.exists(f):
        dataset_path = f
        break

if dataset_path is None:
    raise FileNotFoundError(
        "❌ Dataset file not found! Please upload 'Model_A.csv' or 'lucas_sentinel2_soil_dataset.csv' to the Colab Files folder on the left."
    )

print(f"✅ Loading dataset from: {dataset_path}")
df = pd.read_csv(dataset_path)
print(f"   Total samples: {len(df):,}")
display(df.head(3))


## 🧪 Step 3: Feature Engineering (Physical Spectral Indices)
We calculate 7 diagnostic soil and canopy spectral indices from the Sentinel-2 bands:
- **$NDVI$**: Vegetation vigor ($rac{B08 - B04}{B08 + B04}$)
- **$BSI$**: Bare Soil Index ($rac{(B11 + B04) - (B08 + B02)}{(B11 + B04) + (B08 + B02)}$)
- **$CMR$**: Clay Minerals Ratio ($rac{B11}{B12}$)
- **$FII$**: Ferrous Iron Index ($rac{B04}{B02}$)
- **$CI$**: Carbonate Index / pH proxy ($rac{B12}{B11}$)
- **$NDRE$**: Red Edge Nitrogen proxy ($rac{B08 - B05}{B08 + B05}$)
- **$SBI$**: Soil Brightness Index ($\sqrt{(B04^2 + B03^2 + B08^2) / 3}$)


In [ ]:
eps = 1e-6
b2 = df['B02'].astype(float)
b3 = df['B03'].astype(float)
b4 = df['B04'].astype(float)
b5 = df['B05'].astype(float)
b6 = df['B06'].astype(float)
b7 = df['B07'].astype(float)
b8 = df['B08'].astype(float)
b8a = df['B8A'].astype(float)
b11 = df['B11'].astype(float)
b12 = df['B12'].astype(float)

# Calculate indices if not already present
if 'NDVI' not in df.columns:
    df['NDVI'] = (b8 - b4) / (b8 + b4 + eps)
if 'BSI' not in df.columns:
    df['BSI'] = ((b11 + b4) - (b8 + b2)) / ((b11 + b4) + (b8 + b2) + eps)
if 'Clay_Ratio' not in df.columns:
    df['Clay_Ratio'] = b11 / (b12 + eps)
if 'Ferric_Iron' not in df.columns:
    df['Ferric_Iron'] = b4 / (b2 + eps)
if 'Carbonate_Index' not in df.columns:
    df['Carbonate_Index'] = b12 / (b11 + eps)
if 'NDRE' not in df.columns:
    df['NDRE'] = (b8 - b5) / (b8 + b5 + eps)
if 'Soil_Brightness' not in df.columns:
    df['Soil_Brightness'] = np.sqrt((b4**2 + b3**2 + b8**2) / 3.0)

feature_cols = [
    'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12',
    'NDVI', 'BSI', 'Clay_Ratio', 'Ferric_Iron', 'Carbonate_Index', 'NDRE', 'Soil_Brightness'
]
if 'Elev' in df.columns:
    feature_cols.append('Elev')

print(f"✅ Prepared {len(feature_cols)} features for model training.")


## 🗺️ Step 4: Spatial Block Partitioning (Zero Spatial Leakage)
Points within the same geographic cluster are grouped together so entire regions are held out during cross-validation.


In [ ]:
# Spatial group assignment
if 'NUTS_2' in df.columns:
    spatial_groups = df['NUTS_2'].astype(str)
    print(f"Using administrative regional blocks (NUTS_2): {spatial_groups.nunique()} unique regions.")
else:
    # 2° geographic grid blocks (~200 km blocks)
    lat_col = 'Latitude' if 'Latitude' in df.columns else 'TH_LAT'
    lon_col = 'Longitude' if 'Longitude' in df.columns else 'TH_LONG'
    min_lon, min_lat = df[lon_col].min(), df[lat_col].min()
    grid_x = ((df[lon_col] - min_lon) // 2.0).astype(int)
    grid_y = ((df[lat_col] - min_lat) // 2.0).astype(int)
    spatial_groups = grid_x.astype(str) + "_" + grid_y.astype(str)
    print(f"Generated {spatial_groups.nunique()} spatial grid blocks (2° x 2°).")


## 🚀 Step 5: Train Models with 5-Fold Spatial Cross-Validation

In [ ]:
targets = {
    'pH': 'pH_CaCl2' if 'pH_CaCl2' in df.columns else 'pH',
    'Nitrogen': 'N',
    'Phosphorus': 'P',
    'Potassium': 'K'
}

gkf = GroupKFold(n_splits=5)
models = {}
cv_metrics = {}
all_predictions = {}

os.makedirs("trained_models", exist_ok=True)

for target_name, target_col in targets.items():
    print("=" * 60)
    print(f"Training XGBoost Regressor for: {target_name} ({target_col})")
    print("=" * 60)

    X = df[feature_cols].values
    y = df[target_col].astype(float).values

    fold_r2, fold_rmse, fold_mae = [], [], []
    y_test_all, y_pred_all = [], []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=spatial_groups)):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx], y[test_idx]

        model = xgb.XGBRegressor(
            n_estimators=250,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.5,
            reg_alpha=0.5,
            random_state=42 + fold,
            n_jobs=-1
        )
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)

        r2 = r2_score(y_te, preds)
        rmse = np.sqrt(mean_squared_error(y_te, preds))
        mae = mean_absolute_error(y_te, preds)

        fold_r2.append(r2)
        fold_rmse.append(rmse)
        fold_mae.append(mae)

        y_test_all.extend(y_te)
        y_pred_all.extend(preds)

        print(f"  Fold {fold + 1}: R² = {r2:.3f} | RMSE = {rmse:.3f} | MAE = {mae:.3f}")

    mean_r2 = np.mean(fold_r2)
    mean_rmse = np.mean(fold_rmse)
    mean_mae = np.mean(fold_mae)

    print(f"\n🎯 Mean Spatial CV for {target_name}: R² = {mean_r2:.3f} | RMSE = {mean_rmse:.3f} | MAE = {mean_mae:.3f}\n")

    cv_metrics[target_name] = {
        'R2': mean_r2,
        'RMSE': mean_rmse,
        'MAE': mean_mae
    }
    all_predictions[target_name] = (np.array(y_test_all), np.array(y_pred_all))

    # Fit final production model on full dataset
    final_model = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.5,
        reg_alpha=0.5,
        random_state=42,
        n_jobs=-1
    )
    final_model.fit(X, y)
    models[target_name] = final_model

    # Save model file
    model_file = f"trained_models/xgboost_{target_name}.json"
    final_model.save_model(model_file)
    print(f"💾 Saved production model: {model_file}\n")


## 📊 Step 6: Visual Evaluation & Feature Importance

In [ ]:
# 1. Actual vs Predicted Scatter Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (target_name, (y_true, y_pred)) in enumerate(all_predictions.items()):
    ax = axes[idx]
    # Sample 1000 points for clear visual scatter
    sample_idx = np.random.choice(len(y_true), min(1000, len(y_true)), replace=False)
    
    ax.scatter(y_true[sample_idx], y_pred[sample_idx], alpha=0.3, color='#2563EB', edgecolors='none')
    min_val = min(y_true[sample_idx].min(), y_pred[sample_idx].min())
    max_val = max(y_true[sample_idx].max(), y_pred[sample_idx].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Ideal (y = x)')
    
    m_r2 = cv_metrics[target_name]['R2']
    m_rmse = cv_metrics[target_name]['RMSE']
    ax.set_title(f"{target_name} (Spatial CV R² = {m_r2:.3f}, RMSE = {m_rmse:.3f})", fontsize=12, fontweight='bold')
    ax.set_xlabel("Measured Lab Ground-Truth", fontsize=10)
    ax.set_ylabel("XGBoost Predicted Value", fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# 2. Feature Importance Bar Charts
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (target_name, model) in enumerate(models.items()):
    ax = axes[idx]
    importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
    importances.tail(10).plot(kind='barh', ax=ax, color='#059669')
    ax.set_title(f"Top 10 Feature Importances: {target_name}", fontsize=12, fontweight='bold')
    ax.set_xlabel("F-Score Importance", fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


## 💾 Step 7: Download Trained Production Models

In [ ]:
# Summary Table
summary_df = pd.DataFrame(cv_metrics).T
print("=" * 50)
print("FINAL SPATIAL BLOCK CROSS-VALIDATION SUMMARY:")
print("=" * 50)
display(summary_df)

# To download model files in Colab:
try:
    from google.colab import files
    for t_name in targets.keys():
        files.download(f"trained_models/xgboost_{t_name}.json")
    print("Download initiated for all 4 trained model files.")
except ImportError:
    pass
